# FM Theory: Bessel functions and sidebands

Frequency modulation is the heart of CLM (and of Chowning's classic papers). This notebook makes the theory visible: Bessel functions, sidebands, index, and carrier:modulator ratios.

Sources:
- [FM documentation](https://ccrma.stanford.edu/software/snd/snd/fm.html)
- [CLM oscil](https://ccrma.stanford.edu/software/snd/snd/sndclm.html)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import IPython.display as ipd

import pysndlib.clm as clm

SR = 44100

def spectrum(x, n=4096):
    rl = np.copy(x[:n].astype(float)); im = np.zeros(n)
    clm.mus_fft(rl, im, n, 1)
    clm.mus_rectangular2polar(rl, im)
    return np.abs(rl[:n//2])

def render_fm(fc, mc, index, dur=1.0, name='fm.wav'):
    with clm.Sound(name, channels=1, srate=SR):
        car = clm.make_oscil(fc)
        mod = clm.make_oscil(mc)
        idx = clm.make_env([0, 1, 100, 1], scaler=clm.hz2radians(mc) * index, duration=dur)
        amp = clm.make_env([0, 0, 5, 1, 95, 1, 100, 0], duration=dur)
        for i in range(int(dur * SR)):
            clm.outa(i, clm.env(amp) * clm.oscil(car, clm.oscil(mod) * clm.env(idx)))
    return clm.file2ndarray(name)[0]

## The Bessel function picture

In y = sin(w_c t + I sin(w_m t)), the amplitude of the k-th sideband pair is the Bessel function J_k(I). pysndlib wraps sndlib's Bessel functions as `clm.bes_j0/bes_j1/bes_jn`:

In [ ]:
x = np.linspace(0, 20, 500)
plt.figure(figsize=(10, 3))
plt.plot(x, [clm.bes_j0(v) for v in x], label='J0')
plt.plot(x, [clm.bes_j1(v) for v in x], label='J1')
plt.plot(x, [clm.bes_jn(2, v) for v in x], label='J2')
plt.plot(x, [clm.bes_jn(3, v) for v in x], label='J3')
plt.axhline(0, color='gray', lw=0.5)
plt.legend(); plt.xlabel('modulation index'); plt.title('Bessel sideband weights')
plt.show()

Key facts to see here: J_0(0) = 1 (at zero index only the carrier remains); as the index grows, energy spreads to higher sidebands; each J_k oscillates and can go negative (phase inversions).

## Index: brightness

In [ ]:
low = render_fm(440.0, 440.0, 0.5, name='fm_low.wav')[0]
high = render_fm(440.0, 440.0, 8.0, name='fm_high.wav')[0]

f = np.arange(2048) * SR / 4096
plt.figure(figsize=(14, 3))
plt.plot(f, spectrum(low), alpha=0.7, label='index 0.5')
plt.plot(f, spectrum(high), alpha=0.7, label='index 8.0')
plt.xlim(0, 5000); plt.legend(); plt.title('same pitch, different brightness')
plt.show()

In [ ]:
ipd.Audio('fm_low.wav', rate=SR)

In [ ]:
ipd.Audio('fm_high.wav', rate=SR)

## A moving index: the spectrogram

An index envelope sweeps the sidebands in time -- the signature FM sound ("brass swells"). `matplotlib`'s spectrogram shows it directly:

In [ ]:
sweep = render_fm(440.0, 440.0, 8.0, dur=2.0, name='fm_sweep.wav')[0]
# a rising index envelope
with clm.Sound('fm_sweep.wav', channels=1, srate=SR):
    car = clm.make_oscil(440.0); mod = clm.make_oscil(440.0)
    idx = clm.make_env([0, 0, 100, 1], scaler=clm.hz2radians(440.0) * 8, duration=2.0)
    amp = clm.make_env([0, 0, 5, 1, 95, 1, 100, 0], duration=2.0)
    for i in range(2 * SR):
        clm.outa(i, clm.env(amp) * clm.oscil(car, clm.oscil(mod) * clm.env(idx)))

sw, _ = clm.file2ndarray('fm_sweep.wav')
plt.figure(figsize=(14, 4))
plt.specgram(sw[0], NFFT=2048, Fs=SR, noverlap=512)
plt.ylim(0, 6000); plt.xlabel('time (s)'); plt.ylabel('Hz')
plt.title('index envelope sweeping sidebands open'); plt.show()

In [ ]:
ipd.Audio('fm_sweep.wav', rate=SR)

## Carrier:modulator ratios

Sidebands appear at fc +/- k*fm. Integer ratios give harmonic spectra; irrational ratios give inharmonic (bell-like) ones:

In [ ]:
ratios = [('1:1 (fc=fm)', 440.0, 440.0),
          ('1:2', 440.0, 880.0),
          ('2:3', 587.0, 880.0),
          ('1:sqrt(2) inharmonic', 440.0, 622.0)]
plt.figure(figsize=(14, 7))
for n, (label, fc, mc) in enumerate(ratios):
    data = render_fm(fc, mc, 3.0, name=f'ratio_{n}.wav')[0]
    plt.subplot(len(ratios), 1, n + 1)
    plt.plot(np.arange(2048) * SR / 4096, spectrum(data), lw=0.8)
    plt.xlim(0, 5000); plt.title(label); plt.ylabel('|X|')
plt.tight_layout(); plt.show()

In [ ]:
ipd.Audio('ratio_3.wav', rate=SR)

## Summary

- FM sideband amplitudes are Bessel functions of the modulation index
- Index controls brightness; ratio controls harmonic vs inharmonic
- Envelopes on the index give the classic evolving timbres

Next: [the classic FM instruments](07_FM_Instruments.ipynb)